# DS-Fall - 02 Train Model

This notebook loads `data/processed`, trains the multi-task DS-Fall model, evaluates fall detection and direction classification, and saves models/metrics/figures.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/ds-fall')
# For local debugging, uncomment and adjust:
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQ = PROJECT_ROOT / 'requirements.txt'
print('PROJECT_ROOT =', PROJECT_ROOT)
if not REQ.exists():
    print('WARNING: requirements.txt not found. Check PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
# Install requirements only when core packages are missing.
import importlib
import subprocess
MODULE_CHECKS = [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn'), ('tensorflow', 'tensorflow')]
def _module_ok(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False
missing = [pkg for module, pkg in MODULE_CHECKS if not _module_ok(module)]
if missing and REQ.exists():
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)])

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import make_config
from src.utils.io import ensure_dir, load_pickle
from src.utils.seed import set_seed
from src.models.ds_fall import build_ds_fall_model
from src.training.dataset import make_tf_dataset
from src.training.train import compile_ds_fall_model
from src.training.callbacks import build_callbacks
from src.training.evaluate import (
    evaluate_direction,
    evaluate_fall_detection,
    evaluate_per_dataset,
    plot_confusion_matrix,
    save_metrics_json,
)
from src.data.visualization import plot_imu_window, plot_training_curves

set_seed(42)
config = make_config(PROJECT_ROOT)
for path in [config.models_dir, config.logs_dir, config.metrics_dir, config.figures_dir / 'training']:
    ensure_dir(path)

print('TensorFlow:', tf.__version__)

## Load processed arrays

In [ ]:
processed = config.processed_dir
X = np.load(processed / 'X.npy')
y_fall = np.load(processed / 'y_fall.npy')
y_direction = np.load(processed / 'y_direction.npy')
direction_mask = np.load(processed / 'direction_mask.npy')
metadata = pd.read_csv(processed / 'metadata.csv')
scaler = load_pickle(processed / 'scaler.pkl')

assert X.ndim == 3 and X.shape[1:] == (100, 6), f'Unexpected X shape: {X.shape}'
assert len(X) == len(metadata) == len(y_fall) == len(y_direction) == len(direction_mask)

print('X shape:', X.shape)
print('Scaler:', scaler)
print('\nSplit distribution:')
print(metadata['split'].value_counts())
print('\nDataset distribution:')
print(metadata['dataset'].value_counts())
print('\nFall class distribution:')
print(pd.Series(y_fall).value_counts())
print('\nDirection class distribution, supervised only:')
print(metadata[metadata['direction_supervised'].astype(bool)]['direction_label'].value_counts())
print('\nBITS resampled count:')
print(int(metadata['resampled'].astype(bool).sum()))
display(metadata.head())

## Create train/validation/test datasets

In [ ]:
train_mask = metadata['split'].eq('train').to_numpy()
val_mask = metadata['split'].eq('val').to_numpy()
test_mask = metadata['split'].eq('test').to_numpy()

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
yf_train, yf_val, yf_test = y_fall[train_mask], y_fall[val_mask], y_fall[test_mask]
yd_train, yd_val, yd_test = y_direction[train_mask], y_direction[val_mask], y_direction[test_mask]
dm_train, dm_val, dm_test = direction_mask[train_mask], direction_mask[val_mask], direction_mask[test_mask]
meta_test = metadata[test_mask].reset_index(drop=True)

BATCH_SIZE = 64
train_ds = make_tf_dataset(X_train, yf_train, yd_train, dm_train, batch_size=BATCH_SIZE, shuffle=True, seed=42)
val_ds = make_tf_dataset(X_val, yf_val, yd_val, dm_val, batch_size=BATCH_SIZE, shuffle=False)

print('Train/val/test:', X_train.shape, X_val.shape, X_test.shape)

## Build and train DS-Fall

In [ ]:
model = build_ds_fall_model(input_shape=(100, 6), num_direction_classes=3, show_summary=True)
model = compile_ds_fall_model(model, learning_rate=1e-3)
callbacks = build_callbacks(config.models_dir, config.logs_dir, monitor='val_fall_output_accuracy', patience=10)

history = model.fit(train_ds, validation_data=val_ds, epochs=80, callbacks=callbacks)
model.save(config.models_dir / 'ds_fall_final.keras')
print('Saved final model:', config.models_dir / 'ds_fall_final.keras')

## Training curves

In [ ]:
hist = history.history
train_fig_dir = config.figures_dir / 'training'
plot_training_curves(hist, ['loss', 'val_loss', 'fall_output_loss', 'val_fall_output_loss', 'direction_output_loss', 'val_direction_output_loss'], save_path=train_fig_dir / 'loss_curves.png')
plot_training_curves(hist, ['fall_output_accuracy', 'val_fall_output_accuracy'], save_path=train_fig_dir / 'fall_accuracy_curve.png')
plot_training_curves(hist, ['direction_output_accuracy', 'val_direction_output_accuracy'], save_path=train_fig_dir / 'direction_accuracy_curve.png')

## Evaluate test set

In [ ]:
fall_metrics = evaluate_fall_detection(model, X_test, yf_test)
direction_metrics = evaluate_direction(model, X_test, yd_test, dm_test)
per_dataset_metrics = evaluate_per_dataset(model, X_test, yf_test, yd_test, dm_test, meta_test)

metrics = {
    'fall_detection': fall_metrics,
    'direction_classification': direction_metrics,
    'per_dataset': per_dataset_metrics,
    'bits_preprocessing': '20Hz row-order uniform interpolation to 50Hz',
}
save_metrics_json(config.metrics_dir / 'test_metrics.json', metrics)

print(json.dumps(metrics, indent=2)[:4000])
plot_confusion_matrix(fall_metrics['confusion_matrix'], ['non_fall', 'fall'], 'Fall confusion matrix', save_path=train_fig_dir / 'fall_confusion_matrix.png')
if direction_metrics.get('num_supervised', 0) > 0:
    plot_confusion_matrix(direction_metrics['confusion_matrix'], ['forward', 'backward', 'lateral'], 'Direction confusion matrix', save_path=train_fig_dir / 'direction_confusion_matrix.png')

## Example predictions

In [ ]:
preds = model.predict(X_test[: min(12, len(X_test))], verbose=0)
fall_probs = preds['fall_output'] if isinstance(preds, dict) else preds[0]
dir_probs = preds['direction_output'] if isinstance(preds, dict) else preds[1]
for i in range(min(3, len(X_test))):
    title = f"test_example_{i}_true_fall={yf_test[i]}_pred_fall={int(np.argmax(fall_probs[i]))}_pred_dir={int(np.argmax(dir_probs[i]))}"
    plot_imu_window(X_test[i], title=title, save_path=train_fig_dir / f'example_prediction_{i}.png')

## Optional TFLite export

In [ ]:
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    (config.models_dir / 'ds_fall_float32.tflite').write_bytes(tflite_model)
    print('Saved float32 TFLite')

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_quant = converter.convert()
    (config.models_dir / 'ds_fall_dynamic_range.tflite').write_bytes(tflite_quant)
    print('Saved dynamic range quantized TFLite')
except Exception as exc:
    print('TFLite export skipped:', exc)